# Entrenamiento del modelo clasificador de riesgo IA

En este notebook entrenamos un modelo baseline para clasificar textos en 4 categorías de riesgo del AI Act:
- Inaceptable
- Alto riesgo
- Riesgo limitado
- Riesgo mínimo

Pasos:
1. Carga de datos (train, validation y test)
2. Vectorización con TF-IDF (bigramas, sublinear_tf)
3. Encoding de features estructuradas (category, context, longitud, num_articles)
4. Combinación de features y entrenamiento (LogisticRegression + class_weight='balanced')
5. Evaluación en validación
6. Guardar artefactos (modelo + vectorizador + encoder)

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os

# Localizar src/classifier/ de forma robusta y ajustar cwd al directorio
# de este notebook para que rutas relativas (datasets/, data/, model/) funcionen
# independientemente de desde donde se lance Jupyter/VS Code.
_cwd = os.getcwd()
_candidates = [
    os.path.join(_cwd, "src", "classifier"),
    os.path.abspath(".."),
    os.path.abspath("."),
]
for _p in _candidates:
    if os.path.isfile(os.path.join(_p, "functions.py")):
        if _p not in sys.path:
            sys.path.insert(0, _p)
        # Cambiar cwd al directorio de este notebook
        os.chdir(os.path.join(_p, "classifier_dataset_fusionado"))
        break

import functions  # noqa: E402
functions.MLFLOW_EXPERIMENT = "clasificador_riesgo_dataset_fusionado"
functions._DATASET_TAGS = {"dataset_type": "fusionado", "dataset_source": "eu_ai_act_flagged"}

## 1. Carga de datos

In [3]:
from pathlib import Path
import pandas as pd

data_dir = Path.cwd() / "data" / "processed"

train_path = data_dir / "train.csv"
val_path   = data_dir / "validation.csv"
test_path  = data_dir / "test.csv"

missing_files = [p.name for p in [train_path, val_path, test_path] if not p.exists()]
if missing_files:
    raise FileNotFoundError(
        f"Missing processed files: {missing_files}. "
        "Please run the preprocessing pipeline to generate them "
        "(e.g., feature engineering scripts) before training."
    )

train_df = pd.read_csv(train_path)
val_df   = pd.read_csv(val_path)
test_df  = pd.read_csv(test_path)

X_train_text = train_df["text_final"]
X_val_text   = val_df["text_final"]
X_test_text  = test_df["text_final"]

y_train = train_df["etiqueta"]
y_val   = val_df["etiqueta"]
y_test  = test_df["etiqueta"]

# Columnas estructuradas disponibles tras el preprocesado
CAT_COLS = ["category", "context"]
NUM_COLS = ["longitud", "num_articles"]

print(f"Train: {len(train_df)} muestras | Columnas: {list(train_df.columns)}")
print(f"Validation: {len(val_df)} muestras")
print(f"Test: {len(test_df)} muestras")

Train: 420 muestras | Columnas: ['text_final', 'category', 'context', 'longitud', 'num_articles', 'etiqueta']
Validation: 90 muestras
Test: 90 muestras


## 2. Vectorización TF-IDF

In [4]:
from functions import crear_tfidf

tfidf, X_train_tfidf, X_val_tfidf, X_test_tfidf = crear_tfidf(
    X_train_text, X_val_text, X_test_text,
    max_features=5000,
    ngram_range=(1, 2),
)

Vocabulario TF-IDF: 5000 términos
Forma train: (420, 5000)
Forma validation: (90, 5000)
Forma test: (90, 5000)


## 3. Encoding de features estructuradas

In [5]:
from sklearn.preprocessing import OneHotEncoder
from scipy.sparse import csr_matrix, hstack

# One-hot encoding de category y context.
# IMPORTANTE: fit solo en train para evitar data leakage hacia val/test.
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
cat_train = ohe.fit_transform(train_df[CAT_COLS])
cat_val   = ohe.transform(val_df[CAT_COLS])
cat_test  = ohe.transform(test_df[CAT_COLS])

# Features numéricas como sparse matrix
num_train = csr_matrix(train_df[NUM_COLS].values.astype(float))
num_val   = csr_matrix(val_df[NUM_COLS].values.astype(float))
num_test  = csr_matrix(test_df[NUM_COLS].values.astype(float))

# Combinar TF-IDF + one-hot + numéricas
X_train_final = hstack([X_train_tfidf, cat_train, num_train])
X_val_final   = hstack([X_val_tfidf,   cat_val,   num_val])
X_test_final  = hstack([X_test_tfidf,  cat_test,  num_test])

print(f"Features TF-IDF:       {X_train_tfidf.shape[1]}")
print(f"Features categóricas:  {cat_train.shape[1]}  (category: {len(ohe.categories_[0])} cats, context: {len(ohe.categories_[1])} cats)")
print(f"Features numéricas:    {len(NUM_COLS)}  ({NUM_COLS})")
print(f"Total features final:  {X_train_final.shape[1]}")

Features TF-IDF:       5000
Features categóricas:  22  (category: 9 cats, context: 13 cats)
Features numéricas:    2  (['longitud', 'num_articles'])
Total features final:  5024


## 4. Entrenamiento del modelo (LogisticRegression + class_weight='balanced')

In [6]:
from functions import entrenar_modelo_baseline

# class_weight='balanced' compensa el desbalanceo 6.5× (inaceptable:131 vs riesgo_limitado:20)
modelo = entrenar_modelo_baseline(
    X_train_final, y_train,
    X_val_final,   y_val,
    class_weight="balanced",
)

=== Resultados en VALIDACIÓN ===

                 precision    recall  f1-score   support

    alto_riesgo       0.87      0.65      0.74        20
    inaceptable       0.71      0.81      0.76        31
riesgo_limitado       0.83      0.77      0.80        13
  riesgo_minimo       0.82      0.88      0.85        26

       accuracy                           0.79        90
      macro avg       0.81      0.78      0.79        90
   weighted avg       0.80      0.79      0.79        90

F1-score macro (validación): 0.7881


## 5. Guardar artefactos

In [7]:
import joblib
import os

output_dir = "model"
os.makedirs(output_dir, exist_ok=True)

joblib.dump(modelo, os.path.join(output_dir, "modelo_baseline.joblib"))
joblib.dump(tfidf,  os.path.join(output_dir, "tfidf_vectorizer.joblib"))
joblib.dump(ohe,    os.path.join(output_dir, "ohe_encoder.joblib"))

print("Modelo baseline guardado en:  model/modelo_baseline.joblib")
print("Vectorizador TF-IDF guardado: model/tfidf_vectorizer.joblib")
print("OHE encoder guardado en:      model/ohe_encoder.joblib")

Modelo baseline guardado en:  model/modelo_baseline.joblib
Vectorizador TF-IDF guardado: model/tfidf_vectorizer.joblib
OHE encoder guardado en:      model/ohe_encoder.joblib


In [8]:
# ── MLflow (solo falla esta celda si el servidor no está disponible) ──
from functions import log_mlflow_safe
from sklearn.metrics import f1_score, accuracy_score

y_val_pred = modelo.predict(X_val_final)

try:
    log_mlflow_safe(
        run_name="exp0_baseline",
        params={
            "model_type":         "LogisticRegression",
            "model_max_iter":     2000,
            "class_weight":       "balanced",
            "tfidf_max_features": 5000,
            "tfidf_ngram_range":  "(1, 2)",
            "tfidf_sublinear_tf": True,
            "extra_features":     "category,context,longitud,num_articles",
            "n_features_total":   X_train_final.shape[1],
        },
        metrics={
            "val_f1_macro": round(f1_score(y_val, y_val_pred, average="macro"), 4),
            "val_accuracy": round(accuracy_score(y_val, y_val_pred), 4),
        },
        artifacts=[
            "model/modelo_baseline.joblib",
            "model/tfidf_vectorizer.joblib",
            "model/ohe_encoder.joblib",
        ],
        tags={"experimento": "0", "features": "tfidf+category+context+longitud+num_articles"},
    )
    print("✓ Exp 0 (baseline) registrado en MLflow")
except Exception as e:
    print(f"⚠ MLflow no disponible: {e}")

Password obtenida desde variable de entorno local.
MLflow configurado correctamente → https://18.201.64.41/
⚠ MLflow no disponible: API request to https://18.201.64.41/api/2.0/mlflow/experiments/get-by-name failed with timeout exception HTTPSConnectionPool(host='18.201.64.41', port=443): Max retries exceeded with url: /api/2.0/mlflow/experiments/get-by-name?experiment_name=clasificador_riesgo_dataset_fusionado (Caused by ConnectTimeoutError(<HTTPSConnection(host='18.201.64.41', port=443) at 0x24fdde3e6f0>, 'Connection to 18.201.64.41 timed out. (connect timeout=120)')). To increase the timeout, set the environment variable MLFLOW_HTTP_REQUEST_TIMEOUT (default: 120, type: int) to a larger value.
